# Global Explainability Synthesis: Congruent vs Incongruent Pure

This notebook replaces the earlier deep-only saliency/SHAP sketch with a task-specific evidence synthesis built on the shared `br_analysis.explainability` core. The aim is to compare all model families, identify what they appear to have learned, and separate plausible EEG signal from confounds.

This contrast asks whether EEG contains decodable structure related to conflict between congruent stimulation and incongruent trials that still resolve into a pure percept.

For this task, performance should be interpreted as weak but potentially meaningful evidence for conflict-related structure only when it survives the performance/confound checks below.


In [67]:
# Unified Environment Setup (Local, Colab, Kaggle)
import os
import sys
import warnings
from pathlib import Path

IS_KAGGLE = False
IS_COLAB = (not IS_KAGGLE) and ("google.colab" in sys.modules or "COLAB_GPU" in os.environ)
IN_COLAB = IS_COLAB

if IS_KAGGLE:
    print("Detected Kaggle environment. Installing dependencies...")
    !pip install -q mne seaborn scikit-learn
elif IS_COLAB:
    print("Detected Google Colab environment. Installing dependencies...")
    !pip install -q mne seaborn scikit-learn
    from google.colab import drive
    drive.mount("/content/drive")

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

if IS_KAGGLE:
    dataset_root = None
    for root, dirs, files in os.walk("/kaggle/input"):
        if "br_analysis" in dirs:
            dataset_root = Path(root)
            break
    SRC_DIR = dataset_root if dataset_root else Path("/kaggle/input/nilsduranlaplaza/datasets/tfg-nils-2026-dataset")
elif IS_COLAB:
    SRC_DIR = Path("/content/drive/MyDrive/TFG_Nils_2026/03_Codi")
else:
    SRC_DIR = NOTEBOOK_DIR.parent.parent / "src"
    if not (SRC_DIR / "br_analysis" / "__init__.py").exists():
        SRC_DIR = NOTEBOOK_DIR.parent

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    from IPython import get_ipython
    ipy = get_ipython()
    if ipy is not None:
        ipy.run_line_magic("load_ext", "autoreload")
        ipy.run_line_magic("autoreload", "2")
except Exception:
    pass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from br_analysis import ProjectPaths, detect_project_paths, get_task_config
from br_analysis.explainability import (
    MODEL_COLORS,
    MODEL_LABELS,
    MODEL_ORDER,
    OOF_CACHE_VERSION,
    build_context,
    build_evidence_table,
    convergence_table,
    evidence_pyramid,
    load_oof_predictions,
    plot_convergence,
    plot_evidence_overview,
    save_tables,
    synthesis_text,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
print(f"Using source directory: {SRC_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: [Errno 2] No such file or directory: '/content/TFG/src/br_analysis/config.py'

In [ ]:
TASK_TAG = 'c_vs_ip'
def resolve_project_paths(analysis_name):
    """Resolve the data/output project using local workspace candidates before external Drive defaults."""
    candidates = []
    if IS_COLAB:
        candidates.append(Path("/content/drive/MyDrive/TFG_Nils_2026"))
    for base in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents, Path.cwd(), *Path.cwd().parents]:
        candidates.extend([base / "TFG_Nils_2026", base.parent / "TFG_Nils_2026"])

    seen = set()
    for project_dir in candidates:
        project_dir = Path(project_dir)
        key = str(project_dir).lower()
        if key in seen:
            continue
        seen.add(key)
        analysis_candidates = [p for p in project_dir.glob("04_An*lisi") if p.is_dir()]
        analysis_root = analysis_candidates[0] if analysis_candidates else project_dir / "04_An?lisi"
        if analysis_root.exists():
            fixed_paths = ProjectPaths(
                project_dir=project_dir,
                data_dir=project_dir / "02_Data" / "01_Preprocessed_epochs",
                analysis_root=analysis_root,
                analysis_dir=analysis_root / analysis_name,
                cache_dir=analysis_root / analysis_name / "cache",
                tables_dir=analysis_root / analysis_name / "tables",
                figures_dir=analysis_root / analysis_name / "figures",
            )
            fixed_paths.ensure()
            print(f"Using data project directory: {project_dir}")
            return fixed_paths

    return detect_project_paths(analysis_name, in_colab=IN_COLAB, in_kaggle=IS_KAGGLE)

ANALYSIS_NAME = f"explainability_{TASK_TAG}"
paths = resolve_project_paths(ANALYSIS_NAME)
task = get_task_config(TASK_TAG, paths.analysis_root)
context = build_context(TASK_TAG, paths, task)

print(f"Project directory: {paths.project_dir}")
print(f"Analysis root: {paths.analysis_root}")
print(f"Notebook outputs: {paths.analysis_dir}")
print(f"Task: {context.display_name} ({context.class_0} vs {context.class_1})")
print(f"Expected cache version: {OOF_CACHE_VERSION}")


## 1. Task framing and expected neural hypotheses

The interpretation is deliberately sensor-level. Frontal/theta and posterior/alpha language refers to electrode regions and frequency bands, not source-localized generators.


In [ ]:
print(context.hypothesis)
print(context.interpretation_default)


## 2. Performance evidence and reliability ranking

Explainability is only useful when attached to model evidence. This table combines fresh OOF validation, available final-holdout summaries, attribution availability, confound risk, and a thesis-use recommendation.


In [ ]:
evidence, diagnostics = build_evidence_table(paths, task, TASK_TAG)
pyramid = evidence_pyramid(evidence)
convergence = convergence_table(evidence, paths, task, TASK_TAG)
save_tables(paths, TASK_TAG, evidence, pyramid, convergence)

display_cols = [
    'Display',
    'Family',
    'Mean accuracy',
    'Std accuracy',
    'N subjects',
    'Final holdout accuracy',
    'Chance/permutation status',
    'Attribution availability',
    'Attribution stability',
    'Confound risk level',
    'Thesis-use recommendation',
]

evidence_display = evidence[display_cols].copy()
for col in ['Mean accuracy', 'Std accuracy', 'Final holdout accuracy']:
    evidence_display[col] = evidence_display[col].map(lambda x: np.nan if pd.isna(x) else round(float(x), 4))

display(evidence_display)
print('Saved evidence tables to:', paths.tables_dir)


### Artifact freshness diagnostics

Fresh caches must match the current grouped-development OOF strategy. Missing or stale deep-model artifacts are kept visible instead of silently excluded.


In [ ]:
cache_status = diagnostics['cache_status'].copy()
display(cache_status[['Model', 'OOF cache', 'OOF usable', 'OOF status']])

subjects = diagnostics['subjects']
print(f"Subjects with at least one fresh model artifact: {len(subjects)}")
if len(subjects):
    display(subjects.head())


### Performance overview

The dashed line is nominal chance. Models near chance should not become scientific support merely because their maps look interpretable.


In [ ]:
fig = plot_evidence_overview(
    evidence,
    TASK_TAG,
    save_path=paths.figures_dir / f'explainability_performance_overview_{TASK_TAG}.png',
)
if fig is not None:
    plt.show()


### Cohort discrimination diagnostics

ROC and precision-recall curves are shown only for models with fresh OOF prediction arrays.


In [ ]:
from sklearn.metrics import auc, average_precision_score, precision_recall_curve, roc_curve

model_predictions = load_oof_predictions(paths, task, TASK_TAG)
if not model_predictions:
    print('No fresh OOF prediction arrays available; ROC/PR diagnostics skipped.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=150)
    for model_name, preds in model_predictions.items():
        y_true = preds['y_true']
        y_prob = preds['y_prob']
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        axes[0].plot(
            fpr,
            tpr,
            lw=2,
            color=MODEL_COLORS.get(model_name, '#777777'),
            label=f"{MODEL_LABELS[model_name]} (AUC={auc(fpr, tpr):.3f})",
        )
        axes[1].plot(
            recall,
            precision,
            lw=2,
            color=MODEL_COLORS.get(model_name, '#777777'),
            label=f"{MODEL_LABELS[model_name]} (AP={average_precision_score(y_true, y_prob):.3f})",
        )
    axes[0].plot([0, 1], [0, 1], color='black', linestyle='--', linewidth=1)
    axes[0].set_title('OOF ROC')
    axes[0].set_xlabel('False positive rate')
    axes[0].set_ylabel('True positive rate')
    axes[0].legend(fontsize=8)
    axes[1].set_title('OOF precision-recall')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].legend(fontsize=8)
    sns.despine(fig=fig)
    fig.tight_layout()
    fig.savefig(paths.figures_dir / f'explainability_oof_roc_pr_{TASK_TAG}.png', dpi=150, bbox_inches='tight')
    plt.show()


## 3. Model-family interpretation

Classical models are treated as interpretable evidence sources. Deep models are treated as exploratory diagnostics unless their validation artifacts are fresh and their maps are stable.


In [ ]:
family_view = evidence[[
    'Display',
    'Family',
    'Mean accuracy',
    'Attribution availability',
    'Attribution stability',
    'Attribution note',
    'Confound risk level',
]].copy()
for col in ['Mean accuracy']:
    family_view[col] = family_view[col].map(lambda x: np.nan if pd.isna(x) else round(float(x), 4))
display(family_view)


## 4. Cross-model convergence maps

This section asks whether model evidence converges on the expected sensor-level regions. Classical rows use their explicit feature definitions; deep rows use cached saliency only when present and fresh.


In [ ]:
conv_display = convergence.copy()
for col in ['Reliability weight', 'Frontal/theta-region alignment', 'Posterior/alpha-region alignment', 'Weighted convergence contribution']:
    conv_display[col] = conv_display[col].map(lambda x: round(float(x), 4))
display(conv_display)

fig = plot_convergence(
    convergence,
    TASK_TAG,
    save_path=paths.figures_dir / f'explainability_convergence_{TASK_TAG}.png',
)
plt.show()


## 5. Confound diagnostics

The main confounds are subject identity, temporal drift, class imbalance, cluster-selection circularity, stale caches, and overinterpretation of sensor topographies.


In [ ]:
confound_cols = [
    'Display',
    'OOF usable',
    'OOF status',
    'Final holdout status',
    'Chance/permutation status',
    'Confound risk level',
    'Thesis-use recommendation',
]
display(evidence[confound_cols])

print('Confound reading checklist:')
print('- Near-chance models are weak evidence even when their attribution maps look plausible.')
print('- Subject fingerprints can inflate within-cohort decoding; use grouped/holdout evidence as the minimum bar.')
print('- Cluster-guided models are useful diagnostics but carry circularity risk if cluster selection and evaluation are not independent.')
print('- Topographies are sensor-level summaries, not source-localization claims.')
print('- Deep saliency is model-behavior evidence; stale or missing caches are not thesis evidence.')


## 6. Evidence pyramid

This ranking is the notebook's compact answer to what should be used in the thesis, what is only supportive/exploratory, and what should be regenerated or discarded.


In [ ]:
display(pyramid)


## 7. Final synthesis: what is probably truly important

The conclusion below is generated from the evidence table and convergence diagnostics. It should be edited only if the underlying artifacts are regenerated and the ranking changes.


In [ ]:
print(synthesis_text(context, evidence, convergence))
